In [1]:
import sys
import os

ROOT = os.path.abspath("..")  # adjust if needed
print(ROOT)
OPENPCDET_PATH = '/storage/project/r-gchou3-0/spanse30/OpenPCDet'
print(OPENPCDET_PATH)
# # Put OpenPCDet FIRST
sys.path.insert(0, OPENPCDET_PATH)

# # Import OpenPCDet pcdet
import pcdet
print("Using pcdet from:", pcdet.__file__)

import pickle as pkl

import os
from pathlib import Path
import copy
import numpy as np
import torch

from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.models import build_network, load_data_to_gpu
from pcdet.datasets import build_dataloader
from pcdet.utils import common_utils
import random

import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

import importlib
import helpers_ptt
importlib.reload(helpers_ptt)
import torch.nn.functional as F

from collections import defaultdict

from pcdet.config import cfg, cfg_from_yaml_file
cfg.clear()

CFG_FILE_MF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint_4frames.yaml' 
CKPT_MF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_multiframe_waymo/default/ckpt/checkpoint_epoch_36.pth'  # <- put your ckpt here

logger = common_utils.create_logger()
logger.info(f'Loaded cfg from {CFG_FILE_MF}')

cfg_from_yaml_file(
    CFG_FILE_MF,
    cfg
)

cfg.TAG = 'centerpoint_multiframe'
cfg.EXP_GROUP_PATH = 'waymo_multiframe'

dataset_mf, test_loader_mf, _ = build_dataloader(
    dataset_cfg=cfg.DATA_CONFIG,
    class_names=cfg.CLASS_NAMES,
    batch_size=1,
    dist=False,
    workers=4,
    logger=logger,
    training=False
)

len_test = len(dataset_mf)
logger.info(f'Test set length: {len_test}')

data_iter_mf = iter(test_loader_mf)
batch_dict_mf = next(data_iter_mf)

load_data_to_gpu(batch_dict_mf)

from pcdet.config import cfg, cfg_from_yaml_file
cfg.clear()


CFG_FILE_SF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint.yaml' 
CKPT_SF = '/storage/project/r-gchou3-0/spanse30/OpenPCDet/output/cfgs/custom_models/centerpoint_singleframe_waymo/default/ckpt/checkpoint_epoch_30.pth'  # <- put your ckpt here

cfg_from_yaml_file(CFG_FILE_SF, cfg)
cfg.TAG = Path(CFG_FILE_SF).stem
cfg.EXP_GROUP_PATH = 'centerpoint_waymo_demo'

import os
# cfg.DATA_CONFIG.DATA_PATH = os.path.join(OPENPCDET_PATH, 'data', 'waymo')

logger_sf = common_utils.create_logger()
logger_sf.info(f'Loaded cfg from {CFG_FILE_SF}')


dataset_sf, test_loader_sf, _ = build_dataloader(
    dataset_cfg=cfg.DATA_CONFIG,
    class_names=cfg.CLASS_NAMES,
    batch_size=1,
    dist=False,
    workers=4,
    logger=logger_sf,
    training=False
)

len_test_sf = len(dataset_sf)
logger.info(f'Test set length: {len_test_sf}')


data_iter_sf = iter(test_loader_sf)
batch_dict_sf = next(data_iter_sf)

load_data_to_gpu(batch_dict_sf)



/storage/scratch1/9/spanse30
/storage/project/r-gchou3-0/spanse30/OpenPCDet
Using pcdet from: /storage/project/r-gchou3-0/spanse30/OpenPCDet/pcdet/__init__.py


2026-04-07 16:48:42,596   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint_4frames.yaml
2026-04-07 16:48:42,621   INFO  Loading Waymo dataset
2026-04-07 16:48:44,704   INFO  Total skipped info 0
2026-04-07 16:48:44,704   INFO  Total samples for Waymo dataset: 39987
2026-04-07 16:48:44,705   INFO  Test set length: 39987
2026-04-07 16:48:46,399   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint.yaml
2026-04-07 16:48:46,399   INFO  Loaded cfg from /storage/project/r-gchou3-0/spanse30/OpenPCDet/tools/cfgs/waymo_models/centerpoint.yaml
2026-04-07 16:48:46,404   INFO  Loading Waymo dataset
2026-04-07 16:48:46,404   INFO  Loading Waymo dataset
2026-04-07 16:48:48,200   INFO  Total skipped info 0
2026-04-07 16:48:48,200   INFO  Total skipped info 0
2026-04-07 16:48:48,201   INFO  Total samples for Waymo dataset: 38597
2026-04-07 16:48:48,201   INFO  Total samples for Waymo dataset: 3

In [4]:
helpers_ptt.plot_4frame_sequence(dataset_mf[10000]['points'])

points shape inside plot function: 166111
points shape inside plot function: 165480
points shape inside plot function: 165495
points shape inside plot function: 166585


In [2]:
len(dataset_sf.infos)

38597

In [3]:
len(dataset_mf.infos)

39987

In [4]:
importlib.reload(helpers_ptt)


<module 'helpers_ptt' from '/storage/project/r-gchou3-0/spanse30/OpenPCDet/helpers_ptt.py'>

In [5]:
# Build dict: frame_id -> index
sf_dict = {info['frame_id']: i for i, info in enumerate(dataset_sf.infos)}
mf_dict = {info['frame_id']: i for i, info in enumerate(dataset_mf.infos)}
common_frame_ids = [fid for fid in sf_dict if fid in mf_dict]
frame_ids = [info['frame_id'] for info in dataset_sf.infos]

unique_segments_sf = sorted(list(set(fid.rsplit('_', 1)[0] for fid in frame_ids)))
    
mf_segments = set(info['point_cloud']['lidar_sequence']
                  for info in dataset_mf.infos)
common_segments = mf_segments.intersection(unique_segments_sf)

# mapping: sf index -> mf index
sf_to_mf = {sf_dict[fid]: mf_dict[fid] for fid in common_frame_ids}


segment_to_global = defaultdict(list)

# Mapping: global_idx -> (segment, local_idx)
global_to_segment = {}

for global_idx, info in enumerate(dataset_mf.infos):
    segment = info['point_cloud']['lidar_sequence']
    if segment in common_segments:
        local_idx = len(segment_to_global[segment])  # current position in segment
        segment_to_global[segment].append(global_idx)

        global_to_segment[global_idx] = (segment, local_idx)


segment_counts = defaultdict(int)

for info in dataset_mf.infos:
    segment = info['point_cloud']['lidar_sequence']
    if segment in common_segments: segment_counts[segment] += 1

segment_last_local_idx = {seg: count - 1 for seg, count in segment_counts.items()}


target_frames_global = {}
target_frames_seg = {}

for seg, last_ind in segment_last_local_idx.items():
    toggle = False
    targets_global = {}
    targets_seg = {}
    for i in range(last_ind+1):
        if i==0:
            targets_seg[i] = None
            targets_global[segment_to_global[seg][i]] = None
        else:
            rem = i%32
            # print(rem)
            if(not rem):
                toggle = not toggle
            if toggle:
                targets_seg[i] = 1
                targets_global[segment_to_global[seg][i]] = 1
            else:
                targets_seg[i] = None
                targets_global[segment_to_global[seg][i]] = None
    target_frames_global[seg] = targets_global
    target_frames_seg[seg] = targets_seg


sf_segment_to_global = defaultdict(list)

for global_idx, info in enumerate(dataset_sf.infos):
    seg = info['frame_id'].rsplit('_', 1)[0]
    if seg in common_segments:
        sf_segment_to_global[seg].append(global_idx)
mf_to_sf_global = {}

for mf_global, (seg, local_idx) in global_to_segment.items():
    if seg in sf_segment_to_global and local_idx < len(sf_segment_to_global[seg]):
        sf_global = sf_segment_to_global[seg][local_idx]
        mf_to_sf_global[mf_global] = sf_global

In [6]:
# i = 10000
# j = sf_to_mf[i]

# dataset_sf[i]['frame_id'] == dataset_mf[j]['frame_id']

In [26]:
import pickle
filepath = '/storage/scratch1/9/spanse30/PTT/datasets/global_hard_d/segment-10203656353524179475_7625_000_7645_000_with_camera_labels_d.pkl'
with open(filepath, 'rb') as f:
    data = pickle.load(f)


In [28]:
data[0].keys()

dict_keys(['frame_id', 'pose', 'gt_boxes', 'pred_boxes', 'pred_scores', 'pred_labels', 'points', 'spoof_gt', 'n_spoof_r', 'n_spoof_k'])

In [8]:
past_frames_list = [62, 61, 60, 59]
points_4frames = []
poses = dataset_mf[62]['poses']
pose_lag0 = poses[:4]
pose_lag0_inv = np.linalg.inv(pose_lag0)
for j,frame in enumerate(past_frames_list):
    if(j==0):
        points_4frames.append(data[frame]['points'])
        continue
    # lag = seg_idx - frame
    # print(j, lag)
    pose = poses[0+j*4:4+j*4]

    points_original = data[frame]['points']
    ones = np.ones(points_original.shape[0]).reshape(-1, 1)
    points_hom = np.hstack((points_original[:, :3], ones))

    points_lag0 = points_hom @ pose.T @ pose_lag0_inv.T

    # print(lag*0.1*ones)
    lag = round(j*0.1, 1)
    print(lag)
    points_transformed = np.hstack((points_lag0[:, :-1], points_original[:, 3:-1], lag*ones))

    # print(points_transformed[:, -1])
    points_4frames.append(points_transformed)

0.1
0.2
0.3


In [9]:
points_4frames = np.concatenate(points_4frames, axis = 0)
points_4frames.shape

(506846, 6)

In [10]:
# helpers_ptt.plot_4frame_sequence(points_4frames)

In [11]:
# helpers_ptt.plot_4frame_sequence(dataset_mf[62]['points'])

In [12]:
# dataset_mf[62]['voxels'].shape

# for k in ['voxels', 'voxel_coords', 'voxel_num_points']:
#     dict_mf.pop(k, None)

NameError: name 'dict_mf' is not defined

In [13]:
dict_mf = dataset_mf[62]
for k in ['voxels', 'voxel_coords', 'voxel_num_points']:
    dict_mf.pop(k, None)
print(dict_mf.keys())
dict_mf= helpers_ptt.inject_gt_names(dict_mf, dataset_mf.class_names)
dict_mf   = dataset_mf.prepare_data(dict_mf)

dict_mf['voxels'].shape

dict_keys(['sample_idx', 'poses', 'points', 'frame_id', 'gt_boxes', 'lidar_aug_matrix', 'use_lead_xyz', 'metadata'])


(230868, 5, 6)

In [14]:
print(dict_mf.keys())

dict_keys(['sample_idx', 'poses', 'points', 'frame_id', 'gt_boxes', 'lidar_aug_matrix', 'use_lead_xyz', 'metadata', 'voxels', 'voxel_coords', 'voxel_num_points'])


In [15]:
dict_mf_mod = dataset_mf[62]
dict_mf_mod['points'] = points_4frames
dict_mf_mod= helpers_ptt.inject_gt_names(dict_mf_mod, dataset_mf.class_names)
dict_mf_mod   = dataset_mf.prepare_data(dict_mf_mod)

dict_mf_mod['voxels'].shape

(231720, 5, 6)

In [16]:
dataset_mf[62].keys()

dict_keys(['sample_idx', 'poses', 'points', 'frame_id', 'gt_boxes', 'lidar_aug_matrix', 'use_lead_xyz', 'voxels', 'voxel_coords', 'voxel_num_points', 'metadata'])

In [17]:
import sys

def deep_getsizeof(obj, seen=None):
    if seen is None:
        seen = set()
    
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    seen.add(obj_id)

    size = sys.getsizeof(obj)

    if isinstance(obj, dict):
        size += sum(deep_getsizeof(k, seen) + deep_getsizeof(v, seen) for k, v in obj.items())
    elif isinstance(obj, (list, tuple, set, frozenset)):
        size += sum(deep_getsizeof(i, seen) for i in obj)

    return size

In [18]:

import sys

import sys
import numpy as np
import torch

import sys
import numpy as np
import torch

def deep_getsizeof(obj, seen=None):
    if seen is None:
        seen = set()
    
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    seen.add(obj_id)

    size = sys.getsizeof(obj)

    # --- key fix ---
    if isinstance(obj, np.ndarray):
        return size + obj.nbytes

    if isinstance(obj, torch.Tensor):
        return size + obj.element_size() * obj.nelement()

    # --- recursive ---
    if isinstance(obj, dict):
        size += sum(deep_getsizeof(k, seen) + deep_getsizeof(v, seen) for k, v in obj.items())
    elif isinstance(obj, (list, tuple, set, frozenset)):
        size += sum(deep_getsizeof(i, seen) for i in obj)

    return size


In [19]:
dataset_sf[0].keys()

dict_keys(['sample_idx', 'points', 'frame_id', 'gt_boxes', 'lidar_aug_matrix', 'use_lead_xyz', 'voxels', 'voxel_coords', 'voxel_num_points', 'metadata'])

In [20]:
dataset_mf[0].keys()

dict_keys(['sample_idx', 'poses', 'points', 'frame_id', 'gt_boxes', 'lidar_aug_matrix', 'use_lead_xyz', 'voxels', 'voxel_coords', 'voxel_num_points', 'metadata'])

In [21]:
deep_getsizeof(dict_mf_mod)

111685137

In [22]:
dict_mf_mod['points'] = data[62]['points']
dict_mf_mod['points'].shape

(126723, 6)

In [23]:
deep_getsizeof(dict_mf_mod)*198/(10**9)

13.685158278

In [24]:
deep_getsizeof(data)/(10**9)

1.255842106

In [33]:
import time

In [34]:
# dataset_files = os.listdir(args.dataset)
dataset_files = os.listdir('/storage/scratch1/9/spanse30/PTT/datasets/global_hard_d')


ordered_segments = [info['point_cloud']['lidar_sequence'] for info in dataset_sf.infos]
ordered_segments = list(dict.fromkeys(ordered_segments))  # remove duplicates while preserving order

# map filenames to segment names
file_map = {f.replace('_d.pkl',''): f for f in dataset_files}
# reorder files
dataset_files_ordered = [file_map[s] for s in ordered_segments if s in file_map]


# seg_dataset_file = f"{args.dataset}/{dataset_files_ordered[0]}"
# seg_dataset_file = f"/storage/scratch1/9/spanse30/PTT/datasets/global_hard_d/{dataset_files_ordered[0]}"
output_dir = f"/storage/scratch1/9/spanse30/PTT/datasets_waymo/global_hard_waymo"
os.makedirs(output_dir, exist_ok=True)

# Open the file in read-binary mode ('rb')
with open(seg_dataset_file, 'rb') as file:
    # Use pickle.load() to deserialize the data
    seg_dataset = pkl.load(file)
current_segment = None
segment_frames = []

start = time.perf_counter()
for(frame,batch)  in enumerate(test_loader_mf):
    # if(frame <=197): continue
    segment = batch['frame_id'][0].rsplit("_", 1)[0]
    seg_idx = global_to_segment[frame][1]
    if segment not in common_segments:
        continue
    if current_segment != segment:

        # save previous segment if it exists
        if current_segment is not None:
            output_path = f"{output_dir}/{current_segment}.pkl"
            with open(output_path, "wb") as f:
                pkl.dump(segment_frames, f)
            break
            # print(f"saved {current_segment}")
            # logger.info("Saved preds for : %s", current_segment)
        current_segment = segment
        # print(current_segment)
        segment_frames = []
        # load spoof dataset
        seg_dataset_file = f"/storage/scratch1/9/spanse30/PTT/datasets/global_hard_d/{current_segment}_d.pkl"
        with open(seg_dataset_file, "rb") as f:
            seg_dataset = pkl.load(f)        
        
    assert batch['frame_id'][0] == seg_dataset[seg_idx]['frame_id']
    
#     past_frames_list = [max(seg_idx - i, 0) for i in range(4)]
#     # print(past_frames_list)
#     points_4frames = []
#     poses = dataset_mf[frame]['poses']
#     pose_lag0 = poses[:4]
#     pose_lag0_inv = np.linalg.inv(pose_lag0)
#     for j,frame_idx in enumerate(past_frames_list):
#         # print(j)
#         if(j==0):
#             points_4frames.append(seg_dataset[frame_idx]['points'])
#             continue
#         lag = seg_idx - frame_idx
#         # print(lag)
#         # print(j, lag)
#         pose = poses[0+j*4:4+j*4]

#         points_original = seg_dataset[frame_idx]['points']
#         ones = np.ones(points_original.shape[0]).reshape(-1, 1)
#         points_hom = np.hstack((points_original[:, :3], ones))

#         points_lag0 = points_hom @ pose.T @ pose_lag0_inv.T

#         # print(lag*0.1*ones)
#         # lag = round(j*0.1, 1)
#         # print(lag)
#         points_transformed = np.hstack((points_lag0[:, :-1], points_original[:, 3:-1], lag*ones))

#         # print(points_transformed[:, -1])
#         points_4frames.append(points_transformed)
#     points_4frames = np.concatenate(points_4frames, axis = 0)
    dict_mf_mod = dataset_mf[frame]
    for k in ['voxels', 'voxel_coords', 'voxel_num_points']:
        dict_mf_mod.pop(k, None)
    dict_mf_mod['points'] = seg_dataset[seg_idx]['points']
    dict_mf_mod['pose'] = seg_dataset[seg_idx]['pose']
    dict_mf_mod['spoof_gt'] = seg_dataset[seg_idx]['spoof_gt']
    
    
    # dict_mf_mod = dataset_mf[frame]
    # dict_mf_mod['points'] = points_4frames
    # dict_mf_mod= helpers_ptt.inject_gt_names(dict_mf_mod, dataset_mf.class_names)
    # dict_mf_mod   = dataset_mf.prepare_data(dict_mf_mod)
    # dict_mf_mod['points'] = seg_dataset[seg_idx]['points']
    print(f"processed frame {seg_idx} in segment {current_segment}")
    segment_frames.append(dict_mf_mod)

end = time.perf_counter()
print(f"{end - start:.4f}")

processed frame 0 in segment segment-10203656353524179475_7625_000_7645_000_with_camera_labels
processed frame 1 in segment segment-10203656353524179475_7625_000_7645_000_with_camera_labels
processed frame 2 in segment segment-10203656353524179475_7625_000_7645_000_with_camera_labels
processed frame 3 in segment segment-10203656353524179475_7625_000_7645_000_with_camera_labels
processed frame 4 in segment segment-10203656353524179475_7625_000_7645_000_with_camera_labels
processed frame 5 in segment segment-10203656353524179475_7625_000_7645_000_with_camera_labels
processed frame 6 in segment segment-10203656353524179475_7625_000_7645_000_with_camera_labels
processed frame 7 in segment segment-10203656353524179475_7625_000_7645_000_with_camera_labels
processed frame 8 in segment segment-10203656353524179475_7625_000_7645_000_with_camera_labels
processed frame 9 in segment segment-10203656353524179475_7625_000_7645_000_with_camera_labels
processed frame 10 in segment segment-102036563535

TypeError: list indices must be integers or slices, not str

In [139]:
# seg_idx = 197
past_frames_list = [max(seg_idx - i, 0) for i in range(4)]
# print(past_frames_list)
points_4frames = []
poses = dataset_mf[0]['poses']
pose_lag0 = poses[:4]
pose_lag0_inv = np.linalg.inv(pose_lag0)
for j,frame in enumerate(past_frames_list):
    # print(j)
    if(j==0):
        points_4frames.append(data[frame]['points'])
        continue
    lag = seg_idx - frame
    # print(lag)
    # print(j, lag)
    pose = poses[0+j*4:4+j*4]

    points_original = data[frame]['points']
    ones = np.ones(points_original.shape[0]).reshape(-1, 1)
    points_hom = np.hstack((points_original[:, :3], ones))

    points_lag0 = points_hom @ pose.T @ pose_lag0_inv.T

    # print(lag*0.1*ones)
    # lag = round(j*0.1, 1)
    # print(lag)
    points_transformed = np.hstack((points_lag0[:, :-1], points_original[:, 3:-1], lag*ones))

    # print(points_transformed[:, -1])
    points_4frames.append(points_transformed)

[197, 196, 195, 194]
1
2
3


In [133]:
sum(dataset_mf[1]['points'][:, -1])*10/dataset_mf[1]['points'].shape[0]
# count = 0
# for i in dataset_mf[1]['points'][:, -1]:
#     print(round(i, 1))
#     if round(i, 1) == 0.1: count +=1
# print(count)

0.7474944761122589

In [35]:
dataset_mf[0].keys()

dict_keys(['sample_idx', 'poses', 'points', 'frame_id', 'gt_boxes', 'lidar_aug_matrix', 'use_lead_xyz', 'voxels', 'voxel_coords', 'voxel_num_points', 'metadata'])

In [36]:
dataset_sf[0].keys()

dict_keys(['sample_idx', 'points', 'frame_id', 'gt_boxes', 'lidar_aug_matrix', 'use_lead_xyz', 'voxels', 'voxel_coords', 'voxel_num_points', 'metadata'])

In [38]:
dataset_sf[0]['use_lead_xyz']

True